# 16. Config Controller — runtime mutation with audited snapshots

Long-running agent services hit a config wall. The default model is wrong. The rate limit is too low. The budget cap is leaking through. The lazy answer is an env var change plus a restart — losing every in-flight request, every cached tool handle, every warm connection pool.

`arcllm.config_controller` is the right answer: a tiny, thread-safe runtime controller that holds an immutable `ConfigSnapshot`, atomically swaps to a new snapshot on `patch()`, fires `on_change` callbacks for live consumers, and emits a `TraceRecord` audit event for every mutation.

**You will learn:**

- Why runtime config mutation matters for long-running services
- The frozen `ConfigSnapshot` pattern and what makes it tamper-resistant
- How to register `on_change` callbacks for fan-out to live consumers
- How `patch()` produces a `TraceRecord` and links into the audit chain (cross-ref [14-trace-store](14-trace-store.ipynb))
- Concurrency semantics under `asyncio` — what's atomic, what isn't
- Operator runbook patterns: lift a rate limit, swap a model, rotate a key

**Import path:** `from arcllm.config_controller import ConfigController, ConfigSnapshot`. These types are intentionally *not* re-exported from the top-level `arcllm` namespace — they're an operator-facing affordance, not part of the standard call surface.

## 1. Setup

In [ ]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

Confirm the import path. `ConfigController` and `ConfigSnapshot` live in `arcllm.config_controller` — not on the top-level `arcllm` package — and we verify that on purpose so the rest of the notebook can't drift.

In [ ]:
import arcllm
from arcllm.config_controller import ConfigController, ConfigSnapshot
from arcllm.exceptions import ArcLLMConfigError
from arcllm.trace_store import TraceRecord

assert "ConfigController" not in arcllm.__all__
assert "ConfigSnapshot" not in arcllm.__all__
print("top-level arcllm.__all__ entries:", len(arcllm.__all__))
print("ConfigController exported at top level?", "ConfigController" in arcllm.__all__)
print("actual import path: from arcllm.config_controller import ConfigController, ConfigSnapshot")

## 2. Why runtime config mutation

A federal-grade agent service has properties that make restart-to-reconfigure expensive:

1. **Long-running sessions.** A research agent may have hours of conversation context loaded. Restarting drops it.
2. **Warm provider connections.** HTTP/2 pools, MCP transports, Vault lease timers, NATS subscriptions — all paid-for state.
3. **In-flight tool calls.** Killing the process mid-call leaves audit trails incomplete and partial state on remote systems.
4. **Operator response time.** When a provider rate-limits you in the middle of an exercise, the SOC needs to swap the failover chain in seconds, not minutes.

Env vars + restart fails all four. `ConfigController` solves the problem with a tiny surface:

- `get_snapshot()` — read the current config (lock-free for the caller).
- `patch(updates, actor=...)` — atomic compare-and-swap to a new snapshot.
- `on_change(callback)` — subscribe to mutations.

Every mutation is **audited** as a `TraceRecord` of type `config_change`, so the audit chain (notebook 14) reflects who changed what, when.

In [ ]:
# The contrast: env-var-and-restart loses everything in flight.
# This is the world we're escaping from. Don't do this in production.
import os

os.environ["OLD_WORLD_MODEL"] = "claude-sonnet-4"
current_model = os.environ["OLD_WORLD_MODEL"]
print("reading model from env:", current_model)
print("to change it: SIGTERM the process, edit env, restart, lose state.")

Now the controller way. Mutate without restart, observe the change, keep state.

In [ ]:
ctrl = ConfigController({"model": "claude-sonnet-4"})
print("before:", ctrl.get_snapshot().model)
ctrl.patch({"model": "claude-opus-4"}, actor="sre-on-call")
print("after :", ctrl.get_snapshot().model)
print("process pid unchanged:", os.getpid())

## 3. `ConfigController` basics

Construction takes one positional dict — the initial configuration — plus an optional `on_event` keyword for the audit sink. The controller validates the initial dict against the `ConfigSnapshot` schema; bad input is caught at construction time, not the first mutation.

**Patchable keys** (the operator surface):

| Key | Type | Purpose |
| --- | --- | --- |
| `model` | `str` | Default model id |
| `temperature` | `float` | Sampling temperature |
| `max_tokens` | `int` | Per-response token cap |
| `daily_budget_limit` | `float \| None` | $ cap per day |
| `monthly_budget_limit` | `float \| None` | $ cap per month |
| `failover_chain` | `list[str]` | Ordered failover providers |

In [ ]:
ctrl = ConfigController(
    {
        "model": "claude-sonnet-4",
        "temperature": 0.4,
        "max_tokens": 8192,
        "daily_budget_limit": 50.0,
        "monthly_budget_limit": 1000.0,
        "failover_chain": ["gpt-4o", "claude-haiku"],
    }
)
snap = ctrl.get_snapshot()
print(type(snap).__name__, "=", snap.model_dump())

Field defaults come from the snapshot model. Only `model` is required; everything else has a sensible default — convenient for tests and ad-hoc operators.

In [ ]:
minimal = ConfigController({"model": "gpt-4o-mini"})
print("defaults applied:")
for k, v in minimal.get_snapshot().model_dump().items():
    print(f"  {k:24s} = {v!r}")

Pass an unknown key to the constructor and Pydantic validates upfront. There is no late surprise.

In [ ]:
# Construction-time validation. This raises before anyone reads a snapshot.
from pydantic import ValidationError

try:
    ConfigController({"model": "x", "temperature": "hot"})  # type: ignore[arg-type]
except ValidationError as e:
    print("caught at construction:", e.errors()[0]["msg"])

## 4. `ConfigSnapshot` is immutable

The snapshot is a frozen Pydantic model (`frozen=True` on the model config). Two consequences matter:

1. **Mutating attempts raise.** Code that captured a snapshot reference cannot be tricked into running with a half-mutated value.
2. **Identity is meaningful.** Two `is`-equal snapshots are guaranteed identical. We use this in §5 for the no-op patch optimization.

Frozen Pydantic's mutation guard raises `ValidationError`, which is the right shape for callers — they catch a single Pydantic exception type.

In [ ]:
snap = ConfigSnapshot(model="claude-sonnet-4", temperature=0.5)
print("original temperature:", snap.temperature)

try:
    snap.temperature = 0.99  # type: ignore[misc]
except ValidationError as e:
    print(f"snapshot is immutable: {e.errors()[0]['type']}")

Frozen does *not* mean every nested value is deep-frozen. A `list` field is still a Python list — but the snapshot's *binding* to that list cannot be reassigned. If you mutate the list in place you've broken the contract; the convention is: **treat snapshots as read-only.** `patch()` produces a new snapshot, so consumers should only ever swap references.

In [ ]:
# Demonstrate the *binding* is frozen even if the list is mutable.
snap = ConfigSnapshot(model="claude-sonnet-4", failover_chain=["a", "b"])

try:
    snap.failover_chain = ["c", "d"]  # type: ignore[misc]
except ValidationError as e:
    print("reassignment blocked:", e.errors()[0]["type"])

# In-place list mutation isn't blocked by frozen=True, but is a contract violation:
# always swap snapshots via patch(). We don't demo the in-place mutation here
# because doing so would set a bad example.

Snapshots are also hashable when their fields are hashable, which means you can use them as dict keys or set members for cache invalidation strategies.

In [ ]:
# Hashable when leaf fields are hashable. failover_chain is a list (unhashable),
# so we exclude it for this demo to keep the point clean.
a = ConfigSnapshot(model="claude-sonnet-4", temperature=0.5)
b = ConfigSnapshot(model="claude-sonnet-4", temperature=0.5)
print("equal value, different identity:", a == b, a is b)

## 5. Mutating config — `patch()`

Every mutation goes through `patch(updates, actor=...)`:

1. **Validate keys.** Only the patchable set is accepted; unknown keys raise.
2. **Diff against current.** A no-op patch (every value already equal) returns the same snapshot — `is`-identical — so consumers can short-circuit.
3. **Build the new snapshot.** Pydantic validates the new combined values.
4. **Atomic swap under lock.** No torn reads.
5. **Fan-out callbacks outside the lock.** Callbacks can't deadlock the controller.
6. **Emit the audit `TraceRecord`.**

`actor` is mandatory — every mutation has an attributed identity. This is the hook the audit story plugs into.

In [ ]:
ctrl = ConfigController({"model": "claude-sonnet-4"})
old = ctrl.get_snapshot()

new = ctrl.patch({"temperature": 0.2}, actor="operator-1")

print("old temperature:", old.temperature)
print("new temperature:", new.temperature)
print("controller now returns:", ctrl.get_snapshot().temperature)
print("old snapshot still intact:", old.temperature)
print("distinct snapshot identity:", new is not old)

Patch multiple fields atomically. Either all changes apply or none do — Pydantic validates the *combined* result, so you can't end up with a half-validated state.

In [ ]:
ctrl = ConfigController({"model": "claude-sonnet-4"})
ctrl.patch(
    {"model": "gpt-4o", "temperature": 0.1, "max_tokens": 2048},
    actor="sre-runbook",
)
snap = ctrl.get_snapshot()
print(snap.model_dump())

Reject unknown keys. The set of patchable keys is a security boundary — adding a new patchable field is a deliberate change, not an accident waiting to happen.

In [ ]:
ctrl = ConfigController({"model": "claude-sonnet-4"})
try:
    ctrl.patch({"prod_url": "https://attacker.example"}, actor="badguy")
except ArcLLMConfigError as e:
    print("blocked:", e)

Empty patches are an error too — `patch({}, actor=...)` is almost always a bug and we catch it loudly rather than silently no-op'ing.

In [ ]:
try:
    ctrl.patch({}, actor="test")
except ArcLLMConfigError as e:
    print("blocked:", e)

**No-op patches** — values that already equal the current snapshot — return the *same* snapshot identity. This is how downstream consumers know they don't need to rebuild caches when an operator tries to set a value that's already in effect.

In [ ]:
ctrl = ConfigController({"model": "claude-sonnet-4", "temperature": 0.7})
before = ctrl.get_snapshot()
after = ctrl.patch({"temperature": 0.7}, actor="test")

print("same snapshot identity:", after is before)
print("mutation produced no event (proven in section 7)")

Validation errors surface as `ArcLLMConfigError`, wrapping the underlying Pydantic message. Callers handle one exception type.

In [ ]:
ctrl = ConfigController({"model": "claude-sonnet-4"})
try:
    ctrl.patch({"temperature": "hot"}, actor="test")
except ArcLLMConfigError as e:
    print("validation error wrapped:", str(e)[:80], "...")

## 6. `on_change` callbacks

Live consumers — the LLM router, telemetry exporter, audit sink, dashboard WebSocket fan-out — register callbacks via `on_change(callback)`. Every successful patch (i.e. one that produced an actual change) fires every callback exactly once with the **new** snapshot.

Two design choices to call out:

1. **Callbacks fire outside the lock.** A slow callback can't block the next patch.
2. **No-op patches don't fire callbacks.** This matches the no-op identity guarantee from §5: if the snapshot didn't change, no consumer needs to know.

In [ ]:
snapshots: list[ConfigSnapshot] = []
ctrl = ConfigController({"model": "claude-sonnet-4"})
ctrl.on_change(snapshots.append)

ctrl.patch({"temperature": 0.3}, actor="op")
ctrl.patch({"temperature": 0.5}, actor="op")
ctrl.patch({"max_tokens": 16384}, actor="op")

print("callbacks fired:", len(snapshots))
for i, s in enumerate(snapshots):
    print(f"  #{i}: temperature={s.temperature}, max_tokens={s.max_tokens}")

Multi-callback fan-out — register two unrelated subscribers and verify both fire.

In [ ]:
router_calls: list[ConfigSnapshot] = []
telemetry_calls: list[ConfigSnapshot] = []

ctrl = ConfigController({"model": "claude-sonnet-4"})
ctrl.on_change(router_calls.append)
ctrl.on_change(telemetry_calls.append)

ctrl.patch({"temperature": 0.1}, actor="sre")

assert len(router_calls) == 1
assert len(telemetry_calls) == 1
assert router_calls[0] is telemetry_calls[0]  # same snapshot fan-out
print("both subscribers received the same snapshot identity")

Verify that no-op patches do **not** fire callbacks. This is critical — downstream caches don't get blown away on every retry of the same set call.

In [ ]:
calls: list[ConfigSnapshot] = []
ctrl = ConfigController({"model": "claude-sonnet-4", "temperature": 0.7})
ctrl.on_change(calls.append)

ctrl.patch({"temperature": 0.7}, actor="op")  # no-op
ctrl.patch({"temperature": 0.7}, actor="op")  # no-op

assert calls == [], f"expected no callback fires, got {len(calls)}"
print("no-op patches correctly suppressed callbacks")

**Realistic consumer pattern.** A live router caches a model factory keyed on the model name. When the model changes, the cache gets invalidated. Demonstrate this with a small in-memory scaffold.

In [ ]:
class RouterCache:
    """Pretend live router. Real one would hold http clients."""

    def __init__(self, ctrl: ConfigController) -> None:
        self.factories: dict[str, str] = {}
        self.invalidations = 0
        ctrl.on_change(self._on_change)
        self._absorb(ctrl.get_snapshot())

    def _on_change(self, snap: ConfigSnapshot) -> None:
        self.factories.clear()
        self.invalidations += 1
        self._absorb(snap)

    def _absorb(self, snap: ConfigSnapshot) -> None:
        self.factories[snap.model] = f"<httpx-client for {snap.model}>"


ctrl = ConfigController({"model": "claude-sonnet-4"})
router = RouterCache(ctrl)
print("factories:", router.factories, "invalidations:", router.invalidations)

ctrl.patch({"model": "gpt-4o"}, actor="sre-failover")
print("factories:", router.factories, "invalidations:", router.invalidations)

## 7. Mutation audit trail — `TraceRecord` emission

Every successful `patch()` emits a `TraceRecord` of `event_type="config_change"` via the `on_event` callback. The record carries:

- `provider="system"`, `model="system"` — config events aren't an LLM call
- `event_data.actor` — who you said you were when you called `patch(actor=...)`
- `event_data.changes` — a dict of `{field: {"old": ..., "new": ...}}`

This is the same `TraceRecord` shape that flows through `JSONLTraceStore` ([14-trace-store](14-trace-store.ipynb)), so once you connect a real store, config mutations land in the same hash-chained, tamper-evident audit ledger as every LLM call.

In [ ]:
events: list[TraceRecord] = []
ctrl = ConfigController({"model": "claude-sonnet-4"}, on_event=events.append)

ctrl.patch({"temperature": 0.3}, actor="operator-1")

assert len(events) == 1
rec = events[0]
print("event_type :", rec.event_type)
print("provider   :", rec.provider)
print("actor      :", rec.event_data["actor"])
print("changes    :", rec.event_data["changes"])

Every changed key shows up in the diff, with both `old` and `new` values. Auditors can reconstruct the full sequence of states without storing every snapshot.

In [ ]:
events: list[TraceRecord] = []
ctrl = ConfigController(
    {"model": "claude-sonnet-4", "temperature": 0.7, "max_tokens": 4096},
    on_event=events.append,
)
ctrl.patch(
    {"model": "gpt-4o", "temperature": 0.1, "max_tokens": 8192},
    actor="admin",
)

for key, diff in events[0].event_data["changes"].items():
    print(f"  {key:14s}: {diff['old']} -> {diff['new']}")

No-op patches emit **no** event. Audit log only reflects real changes — the same guarantee as the callbacks.

In [ ]:
events: list[TraceRecord] = []
ctrl = ConfigController(
    {"model": "claude-sonnet-4", "temperature": 0.7},
    on_event=events.append,
)
ctrl.patch({"temperature": 0.7}, actor="test")

assert events == []
print("no events emitted for no-op patch")

**Cross-ref to the chain.** Real audit storage hashes records together so any tampering shows up as a chain break. Demonstrate by feeding the controller's events into the same `with_hash()` linkage logic that `JSONLTraceStore` uses, then verify integrity ourselves.

In [ ]:
# Mini audit chain — same algorithm JSONLTraceStore uses internally.
chained: list[TraceRecord] = []
prev_hash = "0" * 64


def chained_sink(rec: TraceRecord) -> None:
    global prev_hash
    sealed = rec.with_hash(prev_hash)
    chained.append(sealed)
    prev_hash = sealed.record_hash


ctrl = ConfigController({"model": "claude-sonnet-4"}, on_event=chained_sink)
ctrl.patch({"temperature": 0.4}, actor="sre")
ctrl.patch({"max_tokens": 8192}, actor="sre")
ctrl.patch({"failover_chain": ["gpt-4o"]}, actor="sre")

for i, r in enumerate(chained):
    print(
        f"#{i} actor={r.event_data['actor']} hash={r.record_hash[:12]}... prev={r.prev_hash[:12]}..."
    )

Verify the chain — every record's `prev_hash` matches the previous record's `record_hash`, and every `record_hash` recomputes correctly from its content. This is the same check `JSONLTraceStore.verify_chain()` runs across the full audit ledger.

In [ ]:
def verify(records: list[TraceRecord]) -> bool:
    expected_prev = "0" * 64
    for r in records:
        if r.prev_hash != expected_prev:
            return False
        if r.record_hash != r.compute_hash():
            return False
        expected_prev = r.record_hash
    return True


print("chain verifies:", verify(chained))
print("(this is the same algorithm used by JSONLTraceStore.verify_chain)")

## 8. Concurrency safety

`ConfigController` uses a `threading.Lock` for the read-modify-write cycle. Two important consequences for `asyncio` users:

1. **Atomic swap.** Concurrent `patch()` calls can never produce a torn state — every observer sees either the pre-patch snapshot or the post-patch snapshot, never something in between.
2. **No total ordering across patches.** Two simultaneous patches that don't overlap on keys both apply. Two patches that *do* overlap are serialized; the winner is whoever got the lock last.

What's *not* atomic: callback fan-out. Callbacks fire serially after the lock releases, so a callback running for snapshot N might still be in flight when snapshot N+1 commits. Consumers must handle out-of-order delivery if they're ever going to dispatch callback work to a worker pool.

In [ ]:
import asyncio

ctrl = ConfigController({"model": "claude-sonnet-4"})
candidates = ["claude-sonnet-4", "claude-opus-4", "gpt-4o", "gemini-1.5-pro"]


async def race_set(value: str) -> ConfigSnapshot:
    return ctrl.patch({"model": value}, actor=f"writer:{value}")


async def race_all() -> list[ConfigSnapshot]:
    return await asyncio.gather(*[race_set(c) for c in candidates])


# Jupyter has its own running event loop, so use await rather than asyncio.run().
results = await race_all()
final = ctrl.get_snapshot()

print("submitted concurrently:", candidates)
print("final snapshot model  :", final.model)
print("is one of submitted?  :", final.model in candidates)
print("every snapshot is one of the candidates (no torn state):")
print("  unique models seen:", sorted({r.model for r in results}))

**Concurrent non-overlapping keys**: two patches that touch different fields both apply. Final state has both changes.

In [ ]:
ctrl = ConfigController({"model": "claude-sonnet-4"})


async def patch_temp() -> None:
    ctrl.patch({"temperature": 0.99}, actor="writer-A")


async def patch_tokens() -> None:
    ctrl.patch({"max_tokens": 16384}, actor="writer-B")


await asyncio.gather(patch_temp(), patch_tokens())
snap = ctrl.get_snapshot()
print(f"final temperature={snap.temperature}, max_tokens={snap.max_tokens}")
assert snap.temperature == 0.99
assert snap.max_tokens == 16384

**Snapshot identity is monotonic per overlap.** A reader pulling `get_snapshot()` during a flurry of patches always sees a complete, valid configuration. We can assert no reader observed a half-applied state by checking the field set is always one of the legitimate combinations.

In [ ]:
ctrl = ConfigController({"model": "claude-sonnet-4", "temperature": 0.0})


async def writer(temps: list[float]) -> None:
    for t in temps:
        ctrl.patch({"temperature": t}, actor="writer")
        await asyncio.sleep(0)


async def reader(out: list[float]) -> None:
    for _ in range(50):
        out.append(ctrl.get_snapshot().temperature)
        await asyncio.sleep(0)


observed: list[float] = []
await asyncio.gather(
    writer([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]),
    reader(observed),
)
valid = {0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7}
print("observed values subset of valid:", set(observed) <= valid)
print("observed sample:", observed[:8], "...")

## 9. Practical patterns — operator runbook

Three real operations the on-call person runs against a live agent service:

1. **Lift a rate / budget limit.** A long exercise overran the daily budget; raise the cap without dropping the session.
2. **Swap the default model.** A provider went down; failover the default to the secondary while you debug.
3. **Rotate the failover chain.** New evidence about pricing or quality; reorder the providers.

Each one is a single `patch()` plus an `on_event` audit record.

In [ ]:
audit: list[TraceRecord] = []
live = ConfigController(
    {
        "model": "claude-sonnet-4",
        "temperature": 0.4,
        "daily_budget_limit": 50.0,
        "failover_chain": ["gpt-4o", "claude-haiku"],
    },
    on_event=audit.append,
)


def runbook_lift_budget(controller: ConfigController, *, new_cap: float, actor: str) -> None:
    """Operator action: raise daily budget cap during a long exercise."""
    controller.patch({"daily_budget_limit": new_cap}, actor=actor)


def runbook_swap_model(controller: ConfigController, *, new_model: str, actor: str) -> None:
    """Operator action: failover the default model."""
    controller.patch({"model": new_model}, actor=actor)


def runbook_rotate_failover(controller: ConfigController, *, chain: list[str], actor: str) -> None:
    """Operator action: change the failover chain order."""
    controller.patch({"failover_chain": chain}, actor=actor)


runbook_lift_budget(live, new_cap=200.0, actor="sre-on-call:alice")
runbook_swap_model(live, new_model="claude-opus-4", actor="sre-on-call:alice")
runbook_rotate_failover(
    live, chain=["gemini-1.5-pro", "gpt-4o", "claude-haiku"], actor="sre-on-call:bob"
)

snap = live.get_snapshot()
print("final live state:")
for k, v in snap.model_dump().items():
    print(f"  {k:24s} = {v!r}")

Audit chain after the runbook — the SOC has a complete record of what changed, by whom, and when.

In [ ]:
for i, rec in enumerate(audit):
    keys = list(rec.event_data["changes"].keys())
    print(f"#{i}  actor={rec.event_data['actor']:25s}  changes={keys}")

**Wiring into `load_model`.** A common pattern: instantiate the controller before `load_model`, register a callback that swaps the active model id when the snapshot changes, and pass the audit sink as `on_event` so config events land in the same `TraceRecord` stream as LLM call telemetry. Sketch below — the actual `load_model` call is mocked to keep this notebook key-free.

In [ ]:
# Confirm load_model is importable — we don't call it (no API key), but the
# integration shape below is exactly how a live service wires the controller in.
from arcllm.registry import load_model

assert callable(load_model)

all_events: list[TraceRecord] = []
controller = ConfigController(
    {"model": "claude-sonnet-4"},
    on_event=all_events.append,
)

active_model = controller.get_snapshot().model


def reroute(snap: ConfigSnapshot) -> None:
    """In real code: rebuild the load_model() handle keyed on snap.model."""
    global active_model
    active_model = snap.model


controller.on_change(reroute)

# Operator swaps model live — no restart, no lost in-flight calls
controller.patch({"model": "claude-opus-4"}, actor="sre")
print("active model now    :", active_model)
print("audit events emitted:", len(all_events))
print("event[0].event_type :", all_events[0].event_type)
print(
    "(in production: pass on_event=audit.append and trace_store=... to load_model so LLM events join the same chain)"
)

## 10. Summary

**What you learned:**

- `ConfigController` is a thread-safe runtime config holder. It does one thing: atomically swap the current `ConfigSnapshot` for a new one and notify subscribers.
- `ConfigSnapshot` is a frozen Pydantic model. Mutation attempts raise `ValidationError`. Treat them as read-only references; produce new snapshots via `patch()`.
- `patch(updates, actor=...)` validates keys against an explicit allowlist, diff-checks (no-op patches return the same identity), validates the combined result through Pydantic, and atomically swaps under a `threading.Lock`.
- `on_change(callback)` registers a fan-out subscriber. Callbacks fire **outside** the lock, **only** for real changes, with the new snapshot as the argument.
- Every successful mutation emits a `TraceRecord` of `event_type="config_change"` via the `on_event` hook. The record carries the actor and an `{old, new}` diff for every changed field — same shape that flows through `JSONLTraceStore` ([14-trace-store](14-trace-store.ipynb)).
- Concurrent patches under `asyncio` are safe: each swap is atomic, every observer sees a fully-validated snapshot, no torn state. Final state is deterministic *per key*; cross-key it depends on lock acquisition order.
- The operator runbook collapses: lift a budget, swap a model, rotate failover — every action is a single `patch()` with an attributed `actor`, audited by construction.

**Public API surface covered (from `arcllm.config_controller`):**

- `ConfigController(initial: dict, *, on_event=None)`
- `ConfigController.get_snapshot() -> ConfigSnapshot`
- `ConfigController.patch(updates: dict, *, actor: str) -> ConfigSnapshot`
- `ConfigController.on_change(callback: Callable[[ConfigSnapshot], None]) -> None`
- `ConfigSnapshot(model, temperature, max_tokens, daily_budget_limit, monthly_budget_limit, failover_chain)` — frozen Pydantic model
- Cross-package: `arcllm.exceptions.ArcLLMConfigError`, `arcllm.trace_store.TraceRecord`

**Important:** these symbols are *not* exported from the top-level `arcllm` package. Import directly: `from arcllm.config_controller import ConfigController, ConfigSnapshot`.

**Next:**

- [14-trace-store](14-trace-store.ipynb) — where the audit `TraceRecord`s are persisted, hash-chained, and verified.
- [09-telemetry-module](09-telemetry-module.ipynb) — the `on_event` callback the controller emits into is the same one `TelemetryModule` uses for LLM call events.
- [15-queue-circuit-breaker](15-queue-circuit-breaker.ipynb) — pair `ConfigController` with circuit-breaker tuning to live-tune resilience knobs.